# 03. Limpieza · Bronze → Silver

Consulta los controles de `src/limpiar_bookeau.py` (paso `limpiar`): conciliación por archivo, diccionario de columnas, normalización de programas y banderas de calidad. Resultados en `docs/artifacts/limpieza/`; los CSV Silver en `data/plata/`.

No se elimina ningún registro: los estados ambiguos, las encuestas «Inválida» (R02) y las diferencias entre fuentes quedan marcados con banderas.

**Antes de ejecutar:** desde la raíz del proyecto, correr `python3 src/main.py --dummy` (datos ficticios versionados) o `python3 src/main.py` (datos reales, que no se versionan). El cuaderno solo lee los resultados del pipeline en `docs/artifacts/`, `data/plata/` y `data/oro/`. Con datos ficticios los conteos son pequeños y no coinciden con las cifras del informe, que usan los datos reales.


In [1]:
from pathlib import Path
import csv
from html import escape
from IPython.display import display, HTML

PROYECTO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "main.py").is_file()),
    None,
)
if PROYECTO is None:
    raise RuntimeError("Abre el cuaderno desde la carpeta del proyecto o desde notebooks/.")
RESULTADOS = PROYECTO / "docs" / "artifacts" / "limpieza"
if not RESULTADOS.is_dir():
    raise RuntimeError("Faltan los resultados del pipeline. Ejecuta primero desde la raíz: "
                       "python3 src/main.py --dummy  (o sin --dummy si tienes los datos reales)")

def cargar(nombre):
    with (RESULTADOS / nombre).open(newline="", encoding="utf-8") as archivo:
        return list(csv.DictReader(archivo))

def tabla(filas, columnas=None, limite=None):
    if not filas:
        print("Sin registros para esta selección.")
        return
    columnas = columnas or list(filas[0])
    mostrar = filas if limite is None else filas[:limite]
    cabecera = "".join(f"<th>{escape(c)}</th>" for c in columnas)
    cuerpo = "".join(
        "<tr>" + "".join(f"<td>{escape(str(fila[c]))}</td>" for c in columnas) + "</tr>"
        for fila in mostrar
    )
    display(HTML(f"<table><thead><tr>{cabecera}</tr></thead><tbody>{cuerpo}</tbody></table>"))
    print(f"Mostrando {len(mostrar)} de {len(filas)} registros.")


## 1. Conciliación de carga

Los cinco grupos conservaron sus conteos, IDs únicos y vínculos de encuesta. `OK` es un resultado estructural, no una certificación de calidad operativa.


In [2]:
tabla(cargar("controles_carga.csv"))
conciliacion = cargar("conciliacion_archivos.csv")
tabla(conciliacion, limite=15)


grupo,archivo_plata,filas_bronce,filas_plata,filas_eliminadas,id_faltantes,id_distintos,filas_excedentes_id_repetido,encuestas_sin_reserva,vinculos_ambiguos,periodos_diferentes,registros_con_banderas,programas_originales_no_vacios,programas_normalizados_no_vacios,encuestas_incluidas_R02,encuestas_excluidas_R02,filas_con_estado_atendida,conciliacion_estructural
Encuesta Express,data/plata/encuesta_satisfaccion_express.csv,6759,6759,0,0,6759,0,0,0,0,143,131,109,6706,53,6757,OK
Encuesta Normal,data/plata/encuesta_satisfaccion_normal.csv,16379,16379,0,0,16379,0,0,0,0,427,140,119,16268,111,16372,OK
Encuesta Reserva Express,data/plata/encuesta_previa_express.csv,13815,13815,0,0,13815,0,0,0,0,144,143,115,13815,0,7559,OK
Encuesta Reserva Normal IP,data/plata/encuesta_previa_normal_ip.csv,23437,23437,0,0,23437,0,0,0,0,220,122,91,23435,2,14782,OK
Reservas,data/plata/reservas.csv,86562,86562,0,0,86562,0,0,0,0,49,213,174,,,42646,OK


Mostrando 5 de 5 registros.


grupo,archivo_origen,hoja,filas_bronce,filas_plata
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 201620.xlsx,Respuestas,0,0
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 201710.xlsx,Respuestas,781,781
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 201719.xlsx,Respuestas,10,10
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 201720.xlsx,Respuestas,487,487
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 201810.xlsx,Respuestas,498,498
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 201819.xlsx,Respuestas,17,17
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 201820.xlsx,Respuestas,594,594
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 201910.xlsx,Respuestas,614,614
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 201919.xlsx,Respuestas,21,21
Encuesta Express,data/bronce/Bookeau/Encuesta Express/Encuesta Estudiante Express 201920.xlsx,Respuestas,381,381


Mostrando 15 de 114 registros.


## 2. Diccionario de columnas

Los encabezados comunes se homologan; las preguntas conservan su consigna en el diccionario. Leer códigos e IDs como texto y no convertir automáticamente respuestas como NA en nulos.


In [3]:
diccionario = cargar("diccionario_columnas.csv")
GRUPO = "Encuesta Express"
tabla([r for r in diccionario if r["grupo"] == GRUPO])


grupo,columna_original,columna_plata,tipo_lectura_recomendado,regla
Encuesta Express,ID,id_reserva,texto,Nombre canónico; valor original conservado salvo vacío o formato fecha
Encuesta Express,Fecha inicio,fecha_inicio,fecha_hora_sin_zona,Nombre canónico; valor original conservado salvo vacío o formato fecha
Encuesta Express,Fecha fin,fecha_fin,fecha_hora_sin_zona,Nombre canónico; valor original conservado salvo vacío o formato fecha
Encuesta Express,Fecha de llegada,fecha_llegada,fecha_hora_sin_zona,Nombre canónico; valor original conservado salvo vacío o formato fecha
Encuesta Express,Fecha de devolución,fecha_devolucion,fecha_hora_sin_zona,Nombre canónico; valor original conservado salvo vacío o formato fecha
Encuesta Express,Estado de la reserva,estado_reserva,texto,Nombre canónico; valor original conservado salvo vacío o formato fecha
Encuesta Express,Prioritaria,prioritaria_original,texto,Nombre canónico; valor original conservado salvo vacío o formato fecha
Encuesta Express,Categoría,categoria,texto,Nombre canónico; valor original conservado salvo vacío o formato fecha
Encuesta Express,Tipo de horario,tipo_horario,texto,Nombre canónico; valor original conservado salvo vacío o formato fecha
Encuesta Express,Servicio,servicio,texto,Nombre canónico; valor original conservado salvo vacío o formato fecha


Mostrando 33 de 33 registros.


## 3. Normalización de programas

`programa` conserva la etiqueta original y `programa_normalizado` ofrece una escritura comparable. La normalización compacta espacios, convierte a mayúsculas y elimina tildes; no fusiona carreras por interpretación semántica.


In [4]:
programas = cargar("mapa_programas.csv")
tabla([r for r in programas if r["grupo"] == "Reservas" and
       r["programa_original"] != r["programa_normalizado"]], limite=30)


grupo,programa_original,programa_normalizado,filas
Reservas,Biologia,BIOLOGIA,31
Reservas,Biología,BIOLOGIA,19
Reservas,Carpintería,CARPINTERIA,10
Reservas,Ciencia Política,CIENCIA POLITICA,1
Reservas,CupiTaller,CUPITALLER,3
Reservas,DISEÃ‘O,DISEA‘O,5
Reservas,DISEÑO,DISENO,458
Reservas,EXTENSIÓN,EXTENSION,48
Reservas,Economia,ECONOMIA,16
Reservas,Economía,ECONOMIA,40


Mostrando 30 de 48 registros.


## 4. Registros para revisar

Los registros con banderas permanecen en Silver. Se conserva la etiqueta de invalidez de encuesta sin borrar respuestas. Las diferencias con Reservas no se resuelven sobrescribiendo atributos.


In [5]:
tabla(cargar("resumen_banderas.csv"))
# Para localizar casos concretos por archivo, hoja y fila:
# tabla(cargar("casos_revision.csv"), limite=20)


grupo,bandera,registros
Encuesta Express,atributos_difieren_de_reserva,88
Encuesta Express,encuesta_marcada_invalida_en_origen,53
Encuesta Express,revisar_finalizada_sin_llegada,2
Encuesta Normal,atributos_difieren_de_reserva,311
Encuesta Normal,encuesta_marcada_invalida_en_origen,111
Encuesta Normal,revisar_cancelacion_con_llegada,2
Encuesta Normal,revisar_finalizada_sin_llegada,4
Encuesta Reserva Express,atributos_difieren_de_reserva,144
Encuesta Reserva Normal IP,atributos_difieren_de_reserva,218
Encuesta Reserva Normal IP,encuesta_marcada_invalida_en_origen,2


Mostrando 13 de 13 registros.


## 5. Consultar Silver

Las siguientes columnas permiten ver tipos y trazabilidad sin mostrar nombres ni correos. Las preguntas de texto libre se conservan para las siguientes entregas.


In [6]:
with (PROYECTO / "data" / "plata" / "reservas.csv").open(newline="", encoding="utf-8") as archivo:
    lector = csv.DictReader(archivo)
    muestra = [fila for _, fila in zip(range(5), lector)]
tabla(muestra, ["id_reserva", "periodo_origen", "fecha_inicio", "estado_reserva", "estado_reserva_analitico",
                "tipo_horario", "programa_normalizado", "archivo_origen", "fila_excel",
                "banderas_calidad_json"])


id_reserva,periodo_origen,fecha_inicio,estado_reserva,estado_reserva_analitico,tipo_horario,programa_normalizado,archivo_origen,fila_excel,banderas_calidad_json
1,201620,2016-08-19 09:00:00.000000,Cancelada con excusa,Cancelada con excusa,Normal,ISIS,data/bronce/Bookeau/Reservas/Reservas201620.xlsx,2,"[""revisar_cancelacion_con_llegada""]"
2,201620,2016-08-19 09:00:00.000000,Cancelada con excusa,Cancelada con excusa,Normal,,data/bronce/Bookeau/Reservas/Reservas201620.xlsx,3,"[""revisar_cancelacion_con_llegada""]"
3,201620,2016-08-22 09:00:00.000000,Cancelada,Cancelada,Normal,INGENIERIA INDUSTRIAL,data/bronce/Bookeau/Reservas/Reservas201620.xlsx,4,[]
27,201620,2016-08-22 10:00:00.000000,Finalizada,Atendida,Normal,INGENIERIA MECANICA,data/bronce/Bookeau/Reservas/Reservas201620.xlsx,5,[]
197,201620,2016-08-22 10:00:00.000000,Cancelada por sanción,Cancelada por sanción,Normal,INGENIERIA INDUSTRIAL,data/bronce/Bookeau/Reservas/Reservas201620.xlsx,6,[]


Mostrando 5 de 5 registros.


## 6. Reejecutar la limpieza

La siguiente celda regenera los CSV y controles desde los Excel originales. Actualizar la documentación si cambian las reglas o las fuentes.


In [7]:
# import runpy
# import sys
# sys.path.insert(0, str(PROYECTO / "src"))
# runpy.run_path(str(PROYECTO / "src" / "limpiar_bookeau.py"), run_name="__main__")
